# Q1: Where do fires occur, and how large are they?
One red circle per fire with **positive recorded burned hectares**. **Circle area is directly proportional to hectares**: twice the hectares means twice the circle area. Zero hectares produces no point. Small fires can be extremely small. The linear size legend uses ordinary numbers, such as 100 and 1,000. All circles use one color; transparency reduces overlap. Hover shows municipality, date, incident ID and exact hectares.

Use `CleanData.csv`, currently 1998–2022 for records with hectares. Positions use existing `MUNICIPIOS.csv` municipality reference coordinates because supplied fire coordinates contain values far outside Catalonia. These are approximate municipal locations, not ignition points or fire footprints. Records without municipality reference coordinates are excluded and counted.

Run all cells for the interactive Altair map. The saved image is an offline preview.

In [ ]:
import pandas as pd
import altair as alt
import json
from IPython.display import display
alt.data_transformers.disable_max_rows()
fires = pd.read_csv("Clean data/CleanData.csv", parse_dates=["data_incendi"], dtype={"codi_municipi": "string"}).dropna(subset=["total_hectares"])
municipalities = pd.read_csv("dades/MUNICIPIOS.csv", sep=";", encoding="cp1252", dtype="string")
municipalities.index = municipalities.COD_INE.str[:5]
for field, column in [("longitud", "LONGITUD_ETRS89_REGCAN95"), ("latitud", "LATITUD_ETRS89_REGCAN95")]:
    fires[field] = fires.codi_municipi.str.zfill(5).map(pd.to_numeric(municipalities[column].str.replace(",", ".", regex=False)))
print(f"Zero-hectare fires (not plotted): {fires.total_hectares.eq(0).sum()}")
print(f"Excluded unknown locations: {fires.latitud.isna().sum()} of {len(fires):,} fires with hectares.")
fires = fires[fires.total_hectares.gt(0)].dropna(subset=["latitud", "longitud"]).sort_values("total_hectares", ascending=False)
with open("Catalonia.geojson", encoding="utf-8") as f: catalonia = json.load(f)

In [ ]:
outline = alt.Chart(alt.Data(values=catalonia["features"])).mark_geoshape(fill="#f4f3ee", stroke="gray")
dots = alt.Chart(fires).mark_circle(color="#b91c1c", opacity=0.45, strokeWidth=0).encode(
    longitude="longitud:Q", latitude="latitud:Q", size=alt.Size("total_hectares:Q", title="Burned area (ha)", scale=alt.Scale(type="linear", domain=[0, fires.total_hectares.max()], range=[0, 2400]), legend=alt.Legend(values=[100, 1000, 10000], format=",.0f")),
    tooltip=[alt.Tooltip("municipi:N", title="Municipality"), alt.Tooltip("data_incendi:T", title="Date"), alt.Tooltip("total_hectares:Q", title="Burned hectares", format=",.2f"), alt.Tooltip("id_incendi:N", title="Incident")]
)
chart_hect = (outline + dots).project(type="mercator").properties(width=650, height=500, title="Fires with recorded burned area in Catalonia (1998–2022)")
chart_hect

## Fire frequency · All merged records, 1998–2024
One circle per municipality; **circle area is proportional to the number of fires**, not hectares. This includes both GovEsp and GenCat records from the merged dataset, including zero or missing hectares. Hover shows the municipality and count.

Locations use municipality reference coordinates, not exact ignition points. Unmapped locations are reported and excluded from the map. Aggregation removes coincident incident marks, although neighbouring circles may still overlap. The first map describes recorded burned area; this second map describes fire frequency across the full project period. The reduced circle-size range is retained.


In [ ]:
all_fires = pd.read_csv("Clean data/CleanData.csv", parse_dates=["data_incendi"], dtype={"codi_municipi": "string", "codi_comarca": "string"})
all_fires = all_fires[all_fires.data_incendi.dt.year.between(1998, 2024)].copy()
for field, column in [("longitud", "LONGITUD_ETRS89_REGCAN95"), ("latitud", "LATITUD_ETRS89_REGCAN95")]:
    all_fires[field] = all_fires.codi_municipi.str.zfill(5).map(pd.to_numeric(municipalities[column].str.replace(",", ".", regex=False)))
located = all_fires.dropna(subset=["latitud", "longitud"])
print(f"Mapped {len(located):,} of {len(all_fires):,} fires; {len(all_fires) - len(located)} have unknown municipality locations.")
frequency = located.groupby("codi_municipi").agg(municipi=("municipi", "first"), longitud=("longitud", "first"), latitud=("latitud", "first"), count=("id_incendi", "size")).reset_index().sort_values("count", ascending=False)


In [ ]:
frequency_dots = alt.Chart(frequency).mark_circle(color="#b91c1c", opacity=0.5, strokeWidth=0).encode(
    longitude="longitud:Q", latitude="latitud:Q",
    size=alt.Size("count:Q", title="Number of fires", scale=alt.Scale(type="linear", domain=[0, frequency["count"].max()], range=[0, 800]), legend=alt.Legend(format=",.0f")),
    tooltip=[alt.Tooltip("municipi:N", title="Municipality"), alt.Tooltip("count:Q", title="Fires", format=",.0f")]
)
frequency_map = (outline + frequency_dots).project(type="mercator").properties(width=650, height=500, title=alt.TitleParams(text="Q1 · Fire frequency by municipality (1998–2024)", subtitle="GovEsp + GenCat · All fires with known municipality locations"))
frequency_map

## Fire density by comarca and the largest fires

Comarca colour shows the total number of recorded fires during **1998–2024 divided by comarca hectares**. ColorBrewer’s sequential Reds scale uses lighter shades for lower density and darker shades for higher density. The outlined circles remain distinguishable by shape. Counts include fires with missing burned area. The `codi_comarca` column in `CleanData.csv` assigns fires to the same 43-comarca geography used in Q3. Fires without a matching comarca are counted below.

Outlined circles highlight the **10 largest fires with recorded hectares and usable coordinates**, currently GovEsp records from 1998–2022. Change `number_of_fires` to adjust the selection. Circles use reported coordinates, which may include imputed positions. The coordinate check uses Catalonia's bounding box. Each circle is sized to match the reported burned area, with radius `sqrt(hectares × 10,000 / π)` metres. The 128-sided polygon used to draw each circle is about 0.04% smaller than the ideal circle. The circles show equivalent area rather than the actual fire perimeter. A cylindrical equal-area projection makes the circles comparable to the land beneath them. Hover shows each fire's date, identifier and hectares.

Boundaries and 2024 areas reuse the inputs from Q3, recovered from its saved chart data. Areas are published in whole km² and converted to hectares. Sources are the [comarca boundaries](https://services.arcgis.com/fwF8e4aLR6TRWtqM/arcgis/rest/services/comarques_catalunya_etiquetes/FeatureServer/0) and [Idescat areas](https://www.idescat.cat/indicadors/?id=aec&n=15227&t=202400).

In [ ]:
from comarca_maps import (
    comarca_areas_hectares,
    comarca_base_map,
    equal_area,
    load_comarca_boundaries,
)

comarca_boundaries = load_comarca_boundaries(equal_area)
area_ha = comarca_areas_hectares(comarca_boundaries)
fire_comarcas = all_fires.codi_comarca

density = pd.DataFrame([feature["properties"] for feature in comarca_boundaries["features"]])
density = density[["CODICOMAR", "NOMCOMAR"]].rename(columns={"CODICOMAR": "code", "NOMCOMAR": "comarca"})
density["fires"] = density.code.map(fire_comarcas.value_counts()).fillna(0).astype(int)
density["area_ha"] = density.code.map(area_ha)
density["fires_per_ha"] = density.fires / density.area_ha
print(f"Excluded from comarca counts: {(~fire_comarcas.isin(density.code)).sum()} fires with missing or unrecognized comarca codes.")

In [ ]:
import numpy as np

In [ ]:
number_of_fires = 10
recorded_fires = pd.read_csv("Clean data/CleanData.csv", parse_dates=["data_incendi"])
west, south, east, north = comarca_boundaries["bbox"]
usable_coordinates = recorded_fires.longitud.between(west, east) & recorded_fires.latitud.between(south, north)
positive_area = recorded_fires.total_hectares.gt(0)
in_period = recorded_fires.data_incendi.dt.year.between(1998, 2024)
largest_fires = recorded_fires[usable_coordinates & positive_area & in_period].nlargest(number_of_fires, "total_hectares")
print(f"Excluded from circle selection: {(in_period & positive_area & ~usable_coordinates).sum()} fires with missing or out-of-bounds coordinates.")

angles = np.linspace(0, 2 * np.pi, 129)
fire_circles = []
for fire in largest_fires.itertuples():
    x, y = equal_area(fire.longitud, fire.latitud)
    radius = np.sqrt(fire.total_hectares * 10_000 / np.pi)
    ring = np.column_stack([x + radius * np.cos(angles), y + radius * np.sin(angles)]).tolist()
    ring[-1] = ring[0]
    fire_circles.append({
        "type": "Feature",
        "geometry": {"type": "Polygon", "coordinates": [ring]},
        "properties": {
            "incident": fire.id_incendi,
            "municipality": fire.municipi if pd.notna(fire.municipi) else "Unknown municipality",
            "date": fire.data_incendi.strftime("%Y-%m-%d"),
            "hectares": fire.total_hectares,
        },
    })

In [ ]:
density_map = comarca_base_map(comarca_boundaries).transform_lookup(
    lookup="properties.CODICOMAR",
    from_=alt.LookupData(density, "code", ["fires", "area_ha", "fires_per_ha"]),
).encode(
    color=alt.Color("fires_per_ha:Q", title="Fires per hectare", scale=alt.Scale(scheme="reds", zero=True)),
    tooltip=[
        alt.Tooltip("properties.NOMCOMAR:N", title="Comarca"),
        alt.Tooltip("fires:Q", title="Fires", format=",.0f"),
        alt.Tooltip("area_ha:Q", title="Comarca hectares", format=",.0f"),
        alt.Tooltip("fires_per_ha:Q", title="Fires per hectare", format=".5f"),
    ],
)

large_fire_map = alt.Chart(alt.Data(values=fire_circles)).mark_geoshape(
    fill="#fee5d9", fillOpacity=0.55, stroke="#67000d", strokeWidth=1.2
).encode(
    tooltip=[
        alt.Tooltip("properties.incident:N", title="Incident"),
        alt.Tooltip("properties.municipality:N", title="Municipality"),
        alt.Tooltip("properties.date:N", title="Date"),
        alt.Tooltip("properties.hectares:Q", title="Burned hectares", format=",.2f"),
    ]
)

comarca_fire_map = (density_map + large_fire_map).project(type="identity", reflectY=True).properties(
    width=650, height=500,
    title=alt.TitleParams(
        text="Q1 · Fire density by comarca (1998–2024)",
        subtitle=f"Outlined circles: {len(largest_fires)} largest located fires with recorded burned area (1998–2022)",
    ),
)
comarca_fire_map.configure_view(stroke=None)

## Final visualization

Municipality fire frequency and comarca fire density, side by side.

In [ ]:
final_view = alt.hconcat(frequency_map, comarca_fire_map).configure_view(stroke=None)
final_view

## View explanation

We chose two complementary maps to show where fires occur and distinguish their frequency from their size.

The left map shows one circle per municipality, with area proportional to its fire count during 1998–2024. Red connects the marks to the fire theme. Nearby circles inevitably overlap, so we use 50% opacity to keep underlying circles visible. Overlapping circles appear darker, revealing concentrations, although exact counts require tooltips.

The right map shows fires per hectare by comarca. Normalizing by land area prevents large comarques from appearing more affected simply because they contain more land, reinforcing their already greater visual prominence. A sequential red palette represents increasing density through darker shades.

Outlined circles highlight the ten largest fires with recorded hectares and usable coordinates. Their areas represent burned hectares, giving a sense of fire magnitude. Pale fills and dark outlines distinguish them from the background. They show equivalent areas, not actual fire boundaries.

Initially, we plotted one point per fire, with each point’s area matching the fire’s burned area at the map’s scale. However, the largest fires dominated, while most smaller fires were barely visible. The final design shows the overall distribution while highlighting the ten largest eligible fires separately.